# 04 — Monte Carlo (uniform) vs WC / RSS

Independent **uniform** draws on every tolerance interval (a datasheet box is
not a $3\sigma$ Gaussian). Histograms share bin edges. Overlay lines:

| Line | Meaning |
|------|---------|
| black solid | analytical nominal |
| red dashed + fill | hypercube WC min…max (notebook 03) |
| green dash-dot | RSS min…max |
| purple dotted | 5 V goal |

A well-behaved MC cloud sits **inside RSS** and well inside the WC box.
ngspice MC redraws the same passives + $V_{\mathrm{ADJ}}$/$I_{\mathrm{ADJ}}$
and scores $V_{\mathrm{OUT}}$ from an OP. LTspice MC lives in notebook 05
and is overlaid here if its `.log` is already on disk.

## Setup — editable BOM and run knobs

This cell is shared by every notebook. It puts `P5V_Regulator/` on `sys.path`
and builds **one** `CircuitParams` object. Change a resistor, capacitor, or
tolerance here and re-run: analytical, SPICE, WC, and MC all follow this object.

**Plot callouts:** each figure cell has an `ann = [dict(...), ...]` list.
Edit `text`, `xy` (data coords of the arrow tip), and `offset=(dx, dy)` in
**points** to nudge a label. Use `xytext=(x, y)` instead of `offset` for data
coordinates. `arrow=False` for a plain label. `ann = []` hides all callouts.

Tolerances are **fractions** (0.01 = 1 %). Figures go to `results/figures/`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_project_root() -> Path:
    start = Path.cwd().resolve()
    named = Path("Sections/Electronics/Analog_and_Digital/LT3010_5V_Supply")
    candidates = [start, *start.parents, start / named]
    if start.name == "notebooks":
        candidates.insert(0, start.parent)
    for cand in candidates:
        if (cand / "p5v_design" / "__init__.py").is_file():
            return cand
    return start

ROOT = _find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from p5v_design.params import CircuitParams, ToleranceSpec, format_eng, LdoParams
from p5v_design import analysis, schematic, plots, stability
from p5v_design import plots_wc_mc as pwc
from p5v_design.ldo_model import dropout_v, ignd_a, tj_c, psrr_db, vadj_typical_vs_temp
from dataclasses import replace

# --- EDITABLE BOM ---
# Datasheet TA01, "5V Supply with Shutdown" (LT3010/LT3010-5 Rev. J page 1).
# Fixed 5 V part: SENSE tied to OUT (R1 = 0). No external divider (R2 open).
# Capacitors are the 1 µF parts drawn on that figure. Load is the labeled 50 mA.
# VIN is printed as 5.4 V to 80 V; 12 V is the single-point bias inside that range.
R1 = 0.0             # SENSE shorted to OUT
R2 = 1.0e12          # open — no bottom resistor
CFF = 0.0            # no feedforward capacitor on the fixed part
COUT = 1.0e-6
CIN = 1.0e-6
ESR_OUT = 10e-3      # ceramic assumption, not printed on TA01
ESR_IN = 10e-3
RLOAD = 100.0        # 5 V / 50 mA
R_TOL = 0.01         # 1 %
C_TOL = 0.10         # 10 %
VIN_NOM = 12.0
VIN_MIN = 5.4
VIN_MAX = 80.0
ILOAD_OP = 50e-3
N_MC = 2000
N_SPICE_MC = 40
N_LTSPICE_MC = 200
SKIP_SPICE_WC_MC = False
RUN_LTSPICE_BATCH = False
SPICE_WORKERS = None
LTSPICE_WORKERS = None

p = CircuitParams(
    r1=R1, r2=R2, cff=CFF, cout=COUT, cin=CIN,
    esr_out=ESR_OUT, esr_in=ESR_IN, r_load=RLOAD,
    r_tol=ToleranceSpec(R_TOL), c_tol=ToleranceSpec(C_TOL),
)
p = replace(p, op=replace(p.op, vin_nom=VIN_NOM, vin_min=VIN_MIN, vin_max=VIN_MAX,
                           iload_op=ILOAD_OP, r_load=RLOAD))

FIG = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("gain =", p.gain, "  Vout nom =", analysis.vout_dc(p))
print("SPICE_WORKERS =", SPICE_WORKERS, "  N_MC =", N_MC)


gain = 1.0   Vout nom = 5.0
SPICE_WORKERS = None   N_MC = 2000


## Analytical MC at the operate point

Each sample redraws $R_1$, $R_2$, $C$, ESR, $V_{\mathrm{ADJ}}$ (25 °C box),
and $I_{\mathrm{ADJ}}$ (0…max). Seed 42. Cache: `results/mc_analytical/`.

In [2]:
from p5v_design.monte_carlo import sample_metrics, spice_monte_carlo
from p5v_design.worst_case import analytical_worst_case, rss_bounds_many, corners_frame
from p5v_design.ltspice_io import meas_frame
from p5v_design.simulate import find_ngspice

wc = analytical_worst_case(p, over_temp=False)
rss = rss_bounds_many(p)
mc = sample_metrics(p, n=N_MC, seed=42, over_temp=False)
mc.save(ROOT/"results"/"mc_analytical")
display(mc.summary)
print("MC Vout p01…p99", mc.frame.vout.quantile(0.01), mc.frame.vout.quantile(0.99))
print("WC Vout", wc.minimum["vout"], wc.maximum["vout"])
print("fraction of MC outside WC",
      ((mc.frame.vout < wc.minimum["vout"]) | (mc.frame.vout > wc.maximum["vout"])).mean())

,mean,std,min,max,p01,p05,p50,p95,p99
vout,5.000198,4.352742e-02,4.925019,5.074893,4.926302,4.932757,5.001328,5.067658,5.073148
headroom,6.699802,4.352742e-02,6.625107,6.774981,6.626852,6.632342,6.698672,6.767243,6.773698
p_diss,0.371590,2.176371e-03,0.367855,0.375349,0.367943,0.368217,0.371534,0.374962,0.375285
efficiency,0.402204,3.501240e-03,0.396157,0.408212,0.396260,0.396779,0.402295,0.407630,0.408072
iin,0.051800,2.082189e-17,0.051800,0.051800,0.051800,0.051800,0.051800,0.051800,0.051800


MC Vout p01…p99 4.926302440257586 5.073147910628671
WC Vout 4.925 5.075
fraction of MC outside WC 0.0


## Histograms with WC / RSS / target (vout, headroom, dissipation)

Same layout as the comparator $V_H$/$V_L$/$V_{\mathrm{HYS}}$ trio.

In [3]:
ann_h = [dict(text="5 V target", xy=(5.0, 0), offset=(10, 36), arrow=True, color="#9467bd")]
for met in ("vout", "headroom", "p_diss", "efficiency"):
    tgt = 5.0 if met == "vout" else None
    pwc.plot_mc_hist_engines(
        {"analytical": mc.frame}, metric=met,
        wc=(wc.minimum[met], wc.maximum[met]),
        rss=(rss[met]["low"], rss[met]["high"]),
        nominal=wc.nominal[met], target=tgt,
        annotations=ann_h if met == "vout" else [],
        path=FIG/f"04_mc_hist_{met}.png",
        title=f"Analytical MC — {met}",
    )
    plt.show()

cell:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## MC cloud, CDF, vs-run, and p01–p99 range bars

In [4]:
cdf = corners_frame(wc)
pwc.plot_mc_cloud(
    {"analytical": mc.frame},
    x="headroom", y="vout",
    wc_box=(wc.minimum["headroom"], wc.maximum["headroom"], wc.minimum["vout"], wc.maximum["vout"]),
    target_y=5.0, path=FIG/"04_mc_cloud.png",
    title=r"MC cloud  (headroom, $V_{\mathrm{OUT}}$)  + WC box",
)
plt.show()
pwc.plot_cdf(
    {"analytical": mc.frame}, metric="vout",
    wc=(wc.minimum["vout"], wc.maximum["vout"]), target=5.0,
    path=FIG/"04_mc_cdf_vout.png",
)
plt.show()
pwc.plot_mc_vs_run(
    mc.frame, metric="vout", wc=(wc.minimum["vout"], wc.maximum["vout"]),
    target=5.0, engine="analytical", path=FIG/"04_mc_vs_run.png",
)
plt.show()
pwc.plot_percentile_bars(
    {"analytical": mc.frame}, metric="vout",
    annotations=[dict(text="p01–p99 vs 5 V", xy=(5.0, 0), offset=(8, 16), arrow=True)],
    path=FIG/"04_mc_p01_p99.png",
)
plt.show()
pwc.plot_wc_rss_band(
    nom=wc.nominal["vout"], wc=(wc.minimum["vout"], wc.maximum["vout"]),
    rss=(rss["vout"]["low"], rss["vout"]["high"]), target=5.0,
    mc=mc.frame["vout"].to_numpy(),
    path=FIG/"04_mc_on_wc_band.png",
    title="MC rug on WC / RSS band",
)
plt.show()
plots.plot_mc_box(
    {"analytical": mc.frame}, metric="vout", target=5.0,
    path=FIG/"04_mc_box_vout.png", title="Analytical MC box — Vout",
)
plt.show()

cell:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## ngspice MC overlay

Same seed, smaller $N$ (`N_SPICE_MC`). Density histograms share bins with
analytical. The WC/RSS lines stay the **analytical** envelope.

In [5]:
spice = None
HAVE_SPICE = False
try:
    find_ngspice()
    HAVE_SPICE = True
except FileNotFoundError as e:
    print(e)

if HAVE_SPICE and not SKIP_SPICE_WC_MC:
    spice = spice_monte_carlo(
        p, n=N_SPICE_MC, seed=42,
        workdir=ROOT/"results"/"ngspice_mc", max_workers=SPICE_WORKERS,
    )
    display(spice.summary)
    frames = {"analytical": mc.frame, "ngspice": spice.frame.rename(columns={"vout_spice": "vout"})}
    pwc.plot_mc_hist_engines(
        frames, metric="vout",
        wc=(wc.minimum["vout"], wc.maximum["vout"]),
        rss=(rss["vout"]["low"], rss["vout"]["high"]),
        nominal=wc.nominal["vout"], target=5.0,
        path=FIG/"04_mc_hist_an_ng.png",
        title="MC  analytical + ngspice",
    )
    plt.show()
    pwc.plot_cdf(frames, metric="vout", wc=(wc.minimum["vout"], wc.maximum["vout"]),
                 target=5.0, path=FIG/"04_mc_cdf_an_ng.png")
    plt.show()
    pwc.plot_mc_cloud(
        frames, x="headroom", y="vout",
        wc_box=(wc.minimum["headroom"], wc.maximum["headroom"], wc.minimum["vout"], wc.maximum["vout"]),
        target_y=5.0, path=FIG/"04_mc_cloud_an_ng.png",
    )
    plt.show()
    pwc.plot_mc_vs_run(spice.frame.rename(columns={"vout_spice": "vout"}), metric="vout",
                       wc=(wc.minimum["vout"], wc.maximum["vout"]), target=5.0,
                       engine="ngspice", path=FIG/"04_mc_vs_run_ng.png")
    plt.show()
    pwc.plot_percentile_bars(frames, metric="vout", path=FIG/"04_p01_p99_an_ng.png")
    plt.show()
    pwc.plot_span_grouped_bars(
        {
            "WC": {"vout": wc.span("vout"), "headroom": wc.span("headroom")},
            "MC 1–99%": {
                "vout": float(mc.frame.vout.quantile(0.99) - mc.frame.vout.quantile(0.01)),
                "headroom": float(mc.frame.headroom.quantile(0.99) - mc.frame.headroom.quantile(0.01)),
            },
            "RSS": {"vout": 2 * rss["vout"]["delta_rss"], "headroom": 2 * rss["headroom"]["delta_rss"]},
        },
        path=FIG/"04_span_wc_mc_rss.png",
        title="Span: WC box vs MC p01–p99 vs RSS",
    )
    plt.show()
else:
    print("ngspice MC skipped")
    frames = {"analytical": mc.frame}

------------------------------------------------------------------------------


  ngspice MC n=40


  ngspice  jobs=40  workers=8  requested=auto  cpu=8  env P5V_SPICE_WORKERS=''


  queue: mc_0000, mc_0001, mc_0002, mc_0003, mc_0004, mc_0005  +34 more


------------------------------------------------------------------------------


...

  [ngspice MC n=40] done  40 ok / 0 fail / 40 jobs  workers=8  wall=1.79s  cpu=12.87s  speedup=7.18× vs serial


------------------------------------------------------------------------------


,mean,std,min,max
vout,4.999849,0.047309,4.931242,5.069935
vout_spice,4.999799,0.047309,4.931193,5.069884
headroom,6.700151,0.047309,6.630065,6.768758


cell:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## LTspice MC overlay (if notebook 05 has already written a log)

This cell is read-only: it does not launch LTspice. Run notebook 05 (or
`python LTSpice/run_ltspice_batch.py mc`) then re-run this cell.

In [6]:
lt_log = ROOT/"LTSpice"/"MC"/"P5V_ISO_Regulator_MC.log"
if lt_log.is_file():
    lt_mc = meas_frame(lt_log, "vout_op", "vout")
    print("LTspice MC N", len(lt_mc), "min/max", lt_mc.vout.min(), lt_mc.vout.max())
    frames3 = dict(frames)
    frames3["LTspice"] = lt_mc
    pwc.plot_mc_hist_engines(
        frames3, metric="vout",
        wc=(wc.minimum["vout"], wc.maximum["vout"]),
        rss=(rss["vout"]["low"], rss["vout"]["high"]),
        nominal=wc.nominal["vout"], target=5.0,
        path=FIG/"04_mc_hist_three_engines.png",
        title="MC  analytical + ngspice + LTspice",
    )
    plt.show()
    pwc.plot_cdf(frames3, metric="vout", wc=(wc.minimum["vout"], wc.maximum["vout"]),
                 target=5.0, path=FIG/"04_mc_cdf_three.png")
    plt.show()
    pwc.plot_percentile_bars(frames3, metric="vout", path=FIG/"04_p01_p99_three.png")
    plt.show()
    pwc.plot_mc_vs_run(lt_mc, metric="vout", wc=(wc.minimum["vout"], wc.maximum["vout"]),
                       target=5.0, engine="LTspice", path=FIG/"04_mc_vs_run_lt.png")
    plt.show()
    plots.plot_mc_box(frames3, metric="vout", target=5.0, path=FIG/"04_mc_box_three.png",
                      title="MC box — three engines")
    plt.show()
else:
    print("no LTspice MC log yet — run notebook 05")

no LTspice MC log yet — run notebook 05
